# AMOC Spectral Analysis

1) discrete-time spectrum (`lambda_K`) and
2) continuous-time generator spectrum (`lambda_L = log(lambda_K)/dt`).

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "kews" / "__init__.py").is_file())
NOTEBOOK_DIR = REPO_ROOT / "experiments/amoc"
for path in (REPO_ROOT, NOTEBOOK_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
OUTPUT_DIR = REPO_ROOT / "outputs" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
from kews.observables import time_delay


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from matplotlib.lines import Line2D

import xarray as xr
import jax.numpy as jnp

from kews.diagnostics import fit_operator, generator_eigenvalues

LABEL_FONTSIZE = 16
LEGEND_FONTSIZE = 14
TICK_FONTSIZE = 13


## Load AMOC time series


In [ ]:
DATA_DIR = NOTEBOOK_DIR / "data"
SCENARIOS = list('abcdefghij')
TIPPING = {'a', 'i'}
YEAR_MIN, YEAR_MAX = 2015, 2400

DATA_FILE = DATA_DIR / 'amoc_si_mldmax_2Juan.nc'

STATE_VAR = 'amoc48N_ssp245'  # options: 'amoc48N_ssp245', 'amoc26N_ssp245'
MEMBER_LABELS = np.array(list('abcdefghij'))
member_to_idx = {m: i for i, m in enumerate(MEMBER_LABELS)}

with xr.open_dataset(DATA_FILE) as ds:
    years_all = 2015 + ds['time'].values.astype(int)
    state_all = np.asarray(ds[STATE_VAR].values, dtype=np.float32)  # (member, time)

mask_year = (years_all >= YEAR_MIN) & (years_all <= YEAR_MAX)
years = years_all[mask_year]

series = {}
for s in SCENARIOS:
    idx = member_to_idx[s]
    vals = state_all[idx, mask_year]
    label = f'SSP245{s}'
    series[label] = {
        'scenario': s,
        'years': np.asarray(years),
        'vals': np.asarray(vals, dtype=np.float32),
    }

print('Dataset:', DATA_FILE)
print('State variable:', STATE_VAR)
print('Loaded scenarios:', ', '.join(series.keys()))


## Fit EDMD and compute discrete + continuous spectra


In [ ]:
DIM_DELAY = 3
LOW_RANK = DIM_DELAY
DT = 1.0  # annual data

results = {}
for label, data in series.items():
    x = jnp.array(data['vals'])
    out = fit_operator(x, observable=time_delay(DIM_DELAY), rank=LOW_RANK)

    eigvals_K = np.asarray(out['eigvals'])
    eigvals_L = np.asarray(generator_eigenvalues(eigvals_K, dt=DT))

    results[label] = {
        **data,
        'eigvals_K': eigvals_K,
        'eigvals_L': eigvals_L,
    }

print(f'DIM_DELAY={DIM_DELAY}, LOW_RANK={LOW_RANK}, DT={DT}')


## Discrete spectrum


In [ ]:
def collect_eigs(results, TIPPING):
  eigs_c0, eigs_c1 = [], []
  for _, res in results.items():
      lam = np.asarray(res["eigvals_K"], dtype=complex)
      if res["scenario"] in TIPPING:
          eigs_c1.append(lam)
      else:
          eigs_c0.append(lam)
  eigs_c0 = np.concatenate(eigs_c0) if eigs_c0 else np.array([], dtype=complex)
  eigs_c1 = np.concatenate(eigs_c1) if eigs_c1 else np.array([], dtype=complex)
  return eigs_c0, eigs_c1

def annulus_band_params(lam, coverage_min=0.2, coverage_max=0.8, nbins=80):
    """
    Returns (r_peak, r_low, r_high) where r_low..r_high is the radius band
    containing approximately `coverage` of radii around the peak radius.
    """
    if lam.size == 0:
        return None
    r = np.abs(lam)
    r_low = np.quantile(r, coverage_min)
    r_high = np.quantile(r, coverage_max)
    return r_low, r_high

def add_annulus(ax, r_low, r_high, *, color, alpha=0.12, zorder=0):
    if r_high <= r_low:
        return
    t = np.linspace(0, 2*np.pi, 360)
    outer = np.c_[np.cos(t) * r_high, np.sin(t) * r_high]
    inner = np.c_[np.cos(t[::-1]) * r_low, np.sin(t[::-1]) * r_low]
    ring = np.vstack([outer, inner])
    ax.add_patch(
        Polygon(
            ring, closed=True,
            facecolor=color, alpha=alpha,
            edgecolor="none", zorder=zorder
        )
    )
# --------- Plot ---------
fig, ax = plt.subplots(figsize=(6, 5.5))
theta = np.linspace(0, 2*np.pi, 400)
ax.plot(np.cos(theta), np.sin(theta), "k-", lw=0.8, alpha=0.5, zorder=1)
ax.axhline(0, ls='--', color="grey", lw=0.6, alpha=0.6, zorder=1)
ax.axvline(0, ls='--', color="grey", lw=0.6, alpha=0.6, zorder=1)
eigs_c0, eigs_c1 = collect_eigs(results, TIPPING)

# Bands (tune coverage to widen/narrow)
coverage_min = 0.4
coverage_max = 0.6
p0 = annulus_band_params(eigs_c0, coverage_min, coverage_max)
p1 = annulus_band_params(eigs_c1, coverage_min, coverage_max)

# Two annuli (one per class)
if p0 is not None:
    r0_low, r0_high = p0
    add_annulus(ax, r0_low, r0_high, color="C0", alpha=0.5, zorder=0)
if p1 is not None:
    r1_low, r1_high = p1
    add_annulus(ax, r1_low, r1_high, color="C3", alpha=0.5, zorder=0)
    
# Scatter points
for _, res in results.items():
  s = res["scenario"]
  color = "C3" if s in TIPPING else "C0"
  lam = np.asarray(res["eigvals_K"], dtype=complex)
  ax.scatter(
      lam.real, lam.imag,
      color=color,
      s=28, alpha=0.9,
      linewidths=0, edgecolors="none",
      zorder=2
  )

legend_handles = [
  Line2D([0], [0], marker='o', linestyle='None',
         markerfacecolor='C0', markeredgecolor='none', markersize=4,
         label='Non-tipping'),
  Line2D([0], [0], marker='o', linestyle='None',
         markerfacecolor='C3', markeredgecolor='none', markersize=4,
         label='Tipping'),
]
ax.legend(handles=legend_handles, loc="upper left", frameon=False, fontsize=LEGEND_FONTSIZE)
ax.set_xlim([-1.2, 1.2])
ax.set_ylim([-1.2, 1.2])
ax.set_xlabel(r"Re($\lambda_K$)", fontsize=LABEL_FONTSIZE)
ax.set_ylabel(r"Im($\lambda_K$)", fontsize=LABEL_FONTSIZE)
ax.set_aspect("equal")
ax.tick_params(labelsize=TICK_FONTSIZE)

ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/amoc_{STATE_VAR}_disc_spec.png', dpi=300, bbox_inches='tight')
plt.show()


## Continuous generator spectrum


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5))

for _, res in results.items():
    s = res['scenario']
    color = 'C3' if s in TIPPING else 'C0'

    lam_L = res['eigvals_L']
    sigma = np.real(lam_L)
    nu = np.imag(lam_L) / (2.0 * np.pi)  # cycles/year
    valid = np.isfinite(sigma) & np.isfinite(nu)

    ax.scatter(
        sigma[valid],
        nu[valid],
        color=color,
        s=48,
        alpha=0.9,
        linewidths=0,
    )

ax.axhline(0, color='k', lw=0.6, alpha=0.6)
ax.axvline(0, color='k', lw=0.6, alpha=0.6)
legend_handles = [
    Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='C0', markeredgecolor='none', markersize=6, label='Non-tipping'),
    Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='C3', markeredgecolor='none', markersize=6, label='Tipping'),
]
ax.legend(handles=legend_handles, loc='upper left', frameon=False, fontsize=LEGEND_FONTSIZE)
ax.set_xlabel('Re($\lambda_L$) [1/yr]', fontsize=LABEL_FONTSIZE)
ax.set_ylabel('Frequency [cycles/yr]', fontsize=LABEL_FONTSIZE)
ax.tick_params(labelsize=TICK_FONTSIZE)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
rows = []
for label, res in results.items():
  s = res["scenario"]
  group = "tipping" if s in TIPPING else "non_tipping"

  lam_L = np.asarray(res["eigvals_L"])
  sigma = np.real(lam_L)
  omega = np.imag(lam_L)

  # drop constant/invalid mode, then pick slowest-decay mode
  valid = np.isfinite(sigma) & np.isfinite(omega) & (np.abs(lam_L) > 1e-10)
  if not np.any(valid):
      continue
  idx_valid = np.where(valid)[0]
  idx = idx_valid[np.argmin(np.abs(sigma[idx_valid]))]

  sig = float(sigma[idx])
  omg = float(omega[idx])

  e_folding_yr = np.nan if sig >= -1e-10 else 1.0 / abs(sig)
  period_yr = np.nan if abs(omg) <= 1e-10 else 2.0 * np.pi / abs(omg)

  rows.append({
      "scenario": s,
      "group": group,
      "sigma_1_per_yr": sig,
      "e_folding_yr": e_folding_yr,
      "period_yr": period_yr,
  })

df = pd.DataFrame(rows).sort_values("scenario")

summary = (
  df.groupby("group")[["sigma_1_per_yr", "e_folding_yr", "period_yr"]]
    .agg(["mean", "median", "std", "count"])
    .round(3)
)
display(summary)


In [ ]:
# Violin plot: dominant-mode e-folding times by group
vals_tip = df.loc[df["group"] == "tipping", "e_folding_yr"].dropna().to_numpy()
vals_non = df.loc[df["group"] == "non_tipping", "e_folding_yr"].dropna().to_numpy()

plot_data = [vals_tip, vals_non]
labels = ["Tipping", "Non-tipping"]
colors = ["C3", "C0"]

fig, ax = plt.subplots(figsize=(6, 5.5))

parts = ax.violinplot(
    plot_data,
    positions=[1, 2],
    showmeans=True,
    showextrema=True,
    widths=0.8,
)

for body, c in zip(parts["bodies"], colors):
    body.set_facecolor(c)
    body.set_edgecolor(c)
    body.set_alpha(0.35)

for key in ("cmeans", "cmins", "cmaxes", "cbars"):
    if key in parts:
        parts[key].set_color("k")
        parts[key].set_linewidth(1.0)

rng = np.random.default_rng(0)
for i, (vals, c) in enumerate(zip(plot_data, colors), start=1):
    if vals.size == 0:
        continue
    x = i + rng.normal(0.0, 0.03, size=vals.size)
    ax.scatter(x, vals, color=c, s=20, alpha=0.75, zorder=3)

ax.set_xticks([1, 2])
ax.set_xticklabels(labels, fontsize=TICK_FONTSIZE)
ax.set_ylabel(r"e-folding time [yr]", fontsize=LABEL_FONTSIZE)
ax.tick_params(labelsize=TICK_FONTSIZE)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/amoc_{STATE_VAR}_efold.png', dpi=300, bbox_inches='tight')
plt.show()


## Rolling spectrum

One panel: light member trajectories and thick group means (C1 tipping, C0 non-tipping).


In [ ]:
from experiments.indicators import window_size

ROLLING_WL = 0.2
ROLLING_TOP_K = 3
ROLLING_LOW_RANK = DIM_DELAY  # full spectrum for DIM_DELAY=3, plotting top-2 slow modes

rolling = {}
for label, res in results.items():
    years = np.asarray(res['years'])
    vals = np.asarray(res['vals'])
    n = len(vals)
    w = window_size(n, ROLLING_WL)

    sigma_slow = np.full((n, ROLLING_TOP_K), np.nan, dtype=float)

    for start in range(0, n - w + 1):
        end_idx = start + w - 1
        out = fit_operator(
            jnp.array(vals[start:start + w]),
            observable=time_delay(DIM_DELAY),
            rank=ROLLING_LOW_RANK,
        )
        lam_L = np.asarray(generator_eigenvalues(out['eigvals'], dt=DT))
        sigma = np.real(lam_L)
        omega = np.imag(lam_L)

        valid = np.isfinite(sigma) & np.isfinite(omega) & (np.abs(lam_L) > 1e-10)
        idx_valid = np.where(valid)[0]
        if idx_valid.size == 0:
            continue

        order = idx_valid[np.argsort(np.abs(sigma[idx_valid]))]
        for j, idx in enumerate(order[:ROLLING_TOP_K]):
            sigma_slow[end_idx, j] = float(sigma[idx])

    rolling[label] = {
        'scenario': res['scenario'],
        'years': years,
        'sigma_slow': sigma_slow,
        'window_size': int(w),
        'first_window_end_year': int(years[w - 1]),
    }

fig, ax = plt.subplots(figsize=(9.0, 5.0))
base_styles = ['-', '--', ':', '-.']
line_styles = [base_styles[j % len(base_styles)] for j in range(ROLLING_TOP_K)]

# Thick group means for each mode.
years_ref = np.asarray(next(iter(rolling.values()))['years'])
for group_name, color, want_tip in [
    ('tipping', 'C3', True),
    ('non-tipping', 'C0', False),
]:
    member_arrays = [
        d['sigma_slow']
        for d in rolling.values()
        if ((d['scenario'] in TIPPING) == want_tip)
    ]
    if not member_arrays:
        continue

    stack = np.stack(member_arrays, axis=0)  # (n_member, n_time, k)
    finite_count = np.sum(np.isfinite(stack), axis=0)
    sum_sigma = np.nansum(stack, axis=0)
    mean_sigma = np.full(finite_count.shape, np.nan, dtype=float)
    np.divide(sum_sigma, finite_count, out=mean_sigma, where=finite_count > 0)

    for j in range(mean_sigma.shape[1]):
        ax.plot(
            years_ref,
            mean_sigma[:, j],
            color=color,
            lw=2,
            ls=line_styles[j],
            label=f'{group_name} mean $\lambda_{{{j+1}}}$',
        )

first_end = int(next(iter(rolling.values()))['first_window_end_year'])
ax.axvspan(years_ref[0], first_end, color='0.9', alpha=0.5)

# ax.set_title('Rolling top-2 $\mathrm{Re}(\lambda_L)$')
ax.set_xlabel('Year', fontsize=LABEL_FONTSIZE)
ax.set_ylabel('$\mathrm{Re}(\lambda_L)$ [1/yr]', fontsize=LABEL_FONTSIZE)
ax.set_xlim(years_ref[0], years_ref[-1])
ax.tick_params(labelsize=TICK_FONTSIZE)
ax.grid(alpha=0.20)
ax.legend(ncol=2, frameon=False, fontsize=LEGEND_FONTSIZE)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/amoc_{STATE_VAR}_gap.png', dpi=300, bbox_inches='tight')
plt.show()
